# Setting Up and Invoking Custom Guardrails with IBM watsonx.governance

This notebook demonstrates how to create and manage **Custom Guardrails** using the IBM watsonx.governance Guardrails Manager API.


## Prerequisites

Before running this notebook, you need:

### For IBM watsonx.governance Cloud

1. **IBM watsonx.governance Instance**
   - Create one at: https://cloud.ibm.com/catalog/services/watsonxgovernance
   - After creation, get the Service Instance ID from the URL (last UUID)

2. **IBM Cloud API Key**
   - Create at: https://cloud.ibm.com/iam/apikeys
   - Click "Create" and save the API key securely

3. **Inventory ID**
   - UUID format identifier for your inventory
   - Obtained from your watsonx.governance setup see the [documentation](https://www.ibm.com/docs/en/waasfgm?topic=cloud-setting-up-default-inventory)

### For IBM watsonx.governance OnPrem (CPD)

1. **CPD Cluster URL**
   - The URL of your Cloud Pak for Data cluster

2. **CPD Username**
   - Your Cloud Pak for Data username

3. **CPD Authentication**
   - Either CPD API Key or CPD Password (one is required)

4. **CPD Version**
   - The version of your Cloud Pak for Data installation

5. **Inventory ID**
   - UUID format identifier for your inventory

### Common (Optional)

**Custom Detector Endpoint** (for creating custom detectors)
   - URL of your external detection service
   - API key for authentication

## Install Dependencies



In [1]:
import requests
import json

## Configuration

Choose one of the following configuration options based on your deployment:
- **Option 1**: IBM watsonx.governance Cloud
- **Option 2**: IBM watsonx.governance OnPrem (CPD)


### Option 1: IBM watsonx.governance Cloud Configuration

Use this configuration if you are using IBM watsonx.governance as a cloud service.

In [ ]:
# IBM Cloud Configuration - Uncomment and edit these values
SERVICE_INSTANCE_ID = "<EDIT THIS>"
IBM_CLOUD_APIKEY = "<EDIT THIS>"
INVENTORY_ID = "<EDIT THIS>"

### Supported Regions:
# us-south (Dallas), eu-de (Frankfurt), au-syd (Sydney), ca-tor (Toronto), jp-tok (Tokyo)
WATSONX_REGION = "us-south"

# Set deployment type
USE_CPD = False

### Option 2: IBM watsonx.governance OnPrem (CPD) Configuration

Use this configuration if you are using IBM watsonx.governance on Cloud Pak for Data.

In [ ]:
# CPD Configuration - Uncomment and edit these values

# Set deployment type
# USE_CPD = True

if USE_CPD:
    CPD_URL = "<EDIT THIS>"  # e.g., "https://cpd-cpd-instance.apps.example.com"
    CPD_USERNAME = "<EDIT THIS>"
    CPD_APIKEY = "<EDIT THIS>"  # Either CPD_APIKEY or CPD_PASSWORD is required
    # CPD_PASSWORD = "<EDIT THIS>"  # Either CPD_APIKEY or CPD_PASSWORD is required
    CPD_VERSION = "5.3"  # e.g., "5.0"
    INVENTORY_ID = "<EDIT THIS>"
    SERVICE_INSTANCE_ID = "00000000-0000-0000-0000-000000000000"
    VERIFY_SSL = False  # Set to False to disable SSL verification for CPD

In [ ]:
# API Endpoints - Set based on deployment type
if USE_CPD:
    # CPD endpoints
    BASE_URL = CPD_URL
    IAM_URL = f"{CPD_URL}/icp4d-api/v1/authorize"
else:
    # IBM Cloud endpoints
    BASE_URL = "https://api.aiopenscale.cloud.ibm.com"
    IAM_URL = "https://iam.cloud.ibm.com/identity/token"

DETECTORS_URL = f"{BASE_URL}/guardrails_manager/v2/detectors"
POLICIES_URL = f"{BASE_URL}/guardrails_manager/v2/policies"
ENFORCE_URL = f"{BASE_URL}/guardrails_manager/v2/enforce"

## Custom Detector Configuration

Before proceeding with authentication, configure your custom detector payload. This payload defines how your external detection service integrates with watsonx.governance.

### Required User Inputs

You **MUST** customize the following fields in the payload below:

1. **`name`** (string, required)
   - A unique identifier for your detector
   - Cannot reuse names from previously created detectors
   - Example: `"my_custom_profanity_detector"`

2. **`url`** (string, required)
   - The endpoint URL of your external detection service
   - Must be a valid HTTPS URL
   - Example: `"https://your-detector-service.example.com"`

3. **`metadata.api_key`** (string, required)
   - Your custom detector's authentication API key
   - Used to authenticate requests to your detection service
   - Example: `"your-api-key-here"`

4. **`metadata.auth_url`** (string, required if using authentication)
   - The authentication endpoint for your detection service
   - Example: `"https://auth.your-service.example.com"`

### Optional Customizations

- **`description`**: Describe what your detector does
- **`direction`**: Choose where to apply detection (`["input"]`, `["output"]`, or `["input", "output"]`)
- **`actions`**: Specify allowed actions (`["block"]`, `["mask"]`, or both)
- **`detector_properties`**: Define configurable parameters for your detector
- **`parameters`**: Add optional runtime parameters

### Metadata Mapping Fields

The `metadata` section uses template variables to map between watsonx.governance and your detector service:

- **Authentication mapping**:
  - `auth.request.body.apikey`: How to send the API key to your auth service
  - `auth.response.token`: Where to find the token in the auth response

- **Detection mapping**:
  - `detect.request.body.*`: How to structure the detection request
  - `detect.response.*`: How to parse the detection response

**Note**: Keep the template variable syntax (e.g., `{{metadata.api_key}}`, `{{text}}`, `{{token}}`) as-is unless your service uses a different structure.

In [ ]:
custom_detector_payload = {
    # REQUIRED: Unique name for your detector (cannot reuse existing names)
    "name": "<EDIT THIS: your_detector_name>",
    # REQUIRED: Description of what your detector does
    "description": "<EDIT THIS: Description of your custom detector>",
    # REQUIRED: Your detector service endpoint URL
    "url": "<EDIT THIS: https://your-detector-service.example.com>",
    # Authentication method (typically "api_key")
    "auth_method": "api_key",
    # Where to apply detection: "input", "output", or both
    "direction": ["input", "output"],
    # Actions your detector supports
    "actions": ["block", "mask"],
    # Detector properties - customize based on your detector's requirements
    "detector_properties": [
        {"name": "input_data", "type": "text", "default_value": "sample_text"},
        {"name": "input_num", "type": "number", "min": 2, "max": 4, "default_value": 3},
    ],
    # Optional parameters
    "parameters": [{"name": "language", "is_optional": True}],
    # Metadata for authentication and detection mapping
    "metadata": {
        # REQUIRED: Your detector's authentication URL
        "auth_url": "<EDIT THIS: https://auth.your-service.example.com>",
        # Authentication request/response mapping
        "auth.request.body.apikey": "{{metadata.api_key}}",
        "auth.response.token": "{{token}}",
        # Detection request mapping (how to send data to your detector)
        "detect.request.body.text": "{{text}}",
        "detect.request.body.token": "{{token}}",
        # Detection response mapping (how to parse your detector's response)
        # Customize these based on your detector's actual response structure
        "detect.response.detection": "{{response.results[0].type}}",
        "detect.response.detection_type": "{{response.results[0].type}}",
        "detect.response.end": "{{response.results[0].end}}",
        "detect.response.score": "{{response.results[0].score}}",
        "detect.response.start": "{{response.results[0].start}}",
        "detect.response.text": "{{response.results[0].match}}",
    },
}

## Authentication

This section handles authentication for both IBM Cloud and CPD deployments.


In [6]:
def get_cloud_access_token():
    """Generate IBM Cloud access token from API key."""
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json",
    }
    data = {
        "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
        "apikey": IBM_CLOUD_APIKEY,
    }

    try:
        response = requests.post(
            IAM_URL,
            data=data,
            headers=headers,
            verify=globals().get("VERIFY_SSL", True),
        )
        status = response.status_code

        if status == 200:
            print("IBM Cloud token obtained successfully")
            return response.json().get("access_token")

        elif status in (400, 401, 403, 404):
            err = response.json()
            raise Exception(
                f"Auth error {status}: {err.get('error_description', response.text)}"
            )

        elif status == 429:
            raise Exception("Rate limit exceeded.")
        else:
            raise Exception(f"Unexpected status {status}: {response.text}")
    except Exception as e:
        raise Exception(f"IBM Cloud authentication failed: {e}")


def get_cpd_access_token():
    """Generate CPD access token from username and API key or password."""
    headers = {"Content-Type": "application/json", "Accept": "application/json"}

    # Prepare authentication data
    data = {"username": CPD_USERNAME}

    # Use API key if available, otherwise use password
    if "CPD_APIKEY" in globals() and CPD_APIKEY:
        data["api_key"] = CPD_APIKEY
    elif "CPD_PASSWORD" in globals() and CPD_PASSWORD:
        data["password"] = CPD_PASSWORD
    else:
        raise Exception("Either CPD_APIKEY or CPD_PASSWORD must be provided")

    try:
        # Use SSL verification setting
        verify_ssl = globals().get("VERIFY_SSL", True)

        response = requests.post(IAM_URL, json=data, headers=headers, verify=verify_ssl)
        status = response.status_code

        if status == 200:
            print("CPD token obtained successfully")
            return response.json().get("token")

        elif status in (400, 401, 403):
            err = response.json()
            raise Exception(f"Auth error {status}: {err.get('message', response.text)}")
        else:
            raise Exception(f"Unexpected status {status}: {response.text}")
    except Exception as e:
        raise Exception(f"CPD authentication failed: {e}")


# Get access token based on deployment type
if USE_CPD:
    token = get_cpd_access_token()
else:
    token = get_cloud_access_token()

IBM Cloud token obtained successfully


In [7]:
# Setup common headers for all API requests
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}",
    "X-Governance-Instance-Id": SERVICE_INSTANCE_ID,
}

## List Available Custom Detectors


In [ ]:
params = {"inventory_id": INVENTORY_ID}
response = requests.get(
    DETECTORS_URL,
    headers=headers,
    params=params,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 200:
    detectors_data = response.json()
    detectors = detectors_data.get(
        "custom_detectors", []
    )  # To view the built in detectors change "custom_detectors" to "detectors"
    if not detectors:
        print("No detectors found.")
    else:
        print(json.dumps(detectors, indent=2, ensure_ascii=False))
else:
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

No detectors found.


## Create a Custom Detector



Custom detectors allow you to integrate external detection services into watsonx.governance.





### Detector Configuration

- **name**: Unique identifier for your detector
- **url**: Endpoint of your detection service
- **api_key**: Customers custom detector authentication key
- **direction**: Where to apply (`input`, `output`, or both)
- **detector_properties**: Configurable parameters (threshold, mode, etc.)
- **parameters**: Optional runtime parameters

In [ ]:
# Create the detector
params = {"inventory_id": INVENTORY_ID}
response = requests.post(
    DETECTORS_URL,
    headers=headers,
    params=params,
    json=custom_detector_payload,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 201:
    detector_response = response.json()
    detector_id = detector_response["metadata"]["id"]
    detectors_name = detector_response["entity"]["name"]
    # Store the detector_id, and detector_name for later use
    CUSTOM_DETECTOR_ID = detector_id
    CUSTOM_DETECTOR_NAME = detector_response["entity"]["name"]
    print("Custom detector created successfully")
    print(json.dumps(detector_response, indent=2, ensure_ascii=False))
else:
    print(f"Error creating detector: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

Custom detector created successfully
{
  "entity": {
    "id": "e06dbfe2-14e6-46d9-9abd-b2dbd364ad1f",
    "inventory_id": "b377c4a5-0966-4b3b-b740-17f0a309e9fc",
    "name": "custom_pii_detector",
    "description": "Custom detector for pii detection using external service",
    "url": "https://piijson.1344j5rogwud.us-south.codeengine.appdomain.cloud",
    "direction": [
      "input",
      "output"
    ],
    "auth_method": "api_key",
    "actions": [
      "block",
      "mask"
    ],
    "metadata": {
      "auth.request.body.apikey": "{{metadata.api_key}}",
      "auth.response.token": "{{token}}",
      "auth_url": "https://auth.1344j5rogwud.us-south.codeengine.appdomain.cloud",
      "detect.request.body.text": "{{text}}",
      "detect.request.body.token": "{{token}}",
      "detect.response.detection": "{{response.results[0].type}}",
      "detect.response.detection_type": "{{response.results[0].type}}",
      "detect.response.end": "{{response.results[0].end}}",
      "detec

## Create a Policy with Custom Detector
Policies define:
- Which detectors to use
- What actions to take (block or mask)
- Different rules for input vs output
- Configuration for each detector

### Actions
- **block**: Prevent content from being processed
- **mask**: Redact detected content with a character (e.g., `*`)

### Policy Status
- **publish**: Active policy (enforced)
- **draft**: Inactive policy (not enforced)

Note: When using custom detectors in policies, you MUST always include a special property called `custom_detector_id` that references the detector's unique identifier. This is **in addition to** all the properties you defined during detector creation.


In [ ]:
# Policy configuration
policy_payload = {
    "name": "Custom PII Detector",  # This must be unique, if you have used a name before you can not reuse it
    "description": "Policy using custom PII detector with blocking and masking actions",
    "block_message": "Content blocked due to policy violation. Please revise your input.",
    "mask_character": "*",
    "output": [],
    "input": [
        {
            "action": "block",
            "detector": CUSTOM_DETECTOR_NAME,
            "detector_properties": [
                {"name": "custom_detector_inventory_id", "value": INVENTORY_ID},
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},
                {"name": "input_data", "type": "text", "value": "custom_pii"},
                {
                    "name": "input_num",
                    "type": "number",
                    "min": 2,
                    "max": 4,
                    "value": 3,
                },
            ],
        }
    ],
    "policy_status": "publish",
    "tags": ["custom", "pii", "content-safety"],
}

# Create the policy
params = {"inventory_id": INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=policy_payload,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 201:
    policy_response = response.json()
    policy_id = policy_response["metadata"]["id"]
    print("Policy created successfully")
    POLICY_ID = policy_id
else:
    print(f"Error creating policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

Policy created successfully


## List and Retrieve Policies

View all policies and get details about specific ones.


In [ ]:
# List all published policies
params = {
    "inventory_id": INVENTORY_ID,
    "policytype": "publish",  # Options: 'publish', 'draft', 'false' (all)
}
response = requests.get(
    POLICIES_URL,
    headers=headers,
    params=params,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 200:
    policies = response.json()
    policy_list = policies.get("policies", [])
    print(f"Found {len(policy_list)} published policies")
    print(json.dumps(policies, indent=2, ensure_ascii=False))

else:
    print(f"Error listing policies: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

Found 1 published policies
{
  "policies": [
    {
      "entity": {
        "inventory_id": "b377c4a5-0966-4b3b-b740-17f0a309e9fc",
        "id": "970c0d00-375a-45e4-b2e7-de3e82ef265b",
        "name": "Custom PII Detector",
        "description": "Policy using custom PII detector with blocking and masking actions",
        "input": [
          {
            "detector": "custom_pii_detector",
            "detector_properties": null
          }
        ],
        "output": [],
        "tags": [
          "custom",
          "pii",
          "content-safety"
        ],
        "status": {
          "state": "active"
        }
      },
      "metadata": {
        "id": "970c0d00-375a-45e4-b2e7-de3e82ef265b",
        "created_at": "2026-02-24T12:24:53Z",
        "created_by": "IBMid-692000J2ZW",
        "modified_at": "2026-02-24T12:24:53Z",
        "modified_by": "IBMid-692000J2ZW"
      }
    }
  ]
}


## Enforce Policy



In [ ]:
if "POLICY_ID" in locals():
    test_text = "My email is me@test.com"
    enforce_payload = {
        "text": test_text,
        "direction": "input",
        "detectors_properties": {
            CUSTOM_DETECTOR_NAME: {  # This must match the detector name that we created at step "Create a Costume detector"
                "api_key": "<EDIT THIS: your-api-key-here>"
            }
        },
    }
    params = {"inventory_id": INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{POLICY_ID}",
        headers=headers,
        params=params,
        json=enforce_payload,
        verify=globals().get("VERIFY_SSL", True),
    )
    if response.status_code == 200:
        result = response.json()
        print("Enforcement Result:")
        print(f"Overall Status: {result['entity']['status']['overall']}")
        print(
            f"Total Detectors: {result['entity']['status']['summary']['total_detectors']}"
        )
        print(f"Succeeded: {result['entity']['status']['summary']['succeeded']}")
        print(f"Failed: {result['entity']['status']['summary']['failed']}")
    else:
        print(f"Error: {response.status_code}")
        print(json.dumps(response.json(), indent=2))
        raise Exception(
            f"API call failed with status {response.status_code}: {response.text}"
        )

else:
    print("Create a policy first")

Enforcement Result:
Overall Status: success
Total Detectors: 1
Succeeded: 1
Failed: 0


## Update an existing Policy 
This cell demonstrates how to update an existing policy using PUT request
You can modify detector configurations, actions, thresholds, or any policy settings

In [21]:
# You can change fields such as detector settings, actions, block_message, mask_character
updated_policy_payload = {
    "name": "Custom PII Detector",  # you can't change the name, note that the name is the same name we of the policy we created at step "Create a Policy with Custom Detector"
    "description": "Updated policy with modified threshold values and settings",
    "block_message": "Content blocked due to updated policy rules. Please revise your input.",
    "mask_character": "#",
    "output": [],
    "input": [
        {
            "action": "block",  # Note: You can change from "block" to "mask"
            "detector": CUSTOM_DETECTOR_NAME,
            "detector_properties": [
                {"name": "custom_detector_inventory_id", "value": INVENTORY_ID},
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},
                {"name": "input_data", "type": "text", "value": "custom_pii"},
                {
                    "name": "input_num",
                    "type": "number",
                    "min": 2,
                    "max": 4,
                    "value": 3,
                },
            ],
        }
    ],
    "policy_status": "publish",
    "tags": [
        "custom",
        "profanity",
        "content-safety",
        "updated_v2",
    ],  # You can add a tag to point out that it was updated
}

if "POLICY_ID" not in locals():
    print("No policy found to modify. Please create a policy first.")
else:
    params = {"inventory_id": INVENTORY_ID}
    response = requests.put(
        f"{POLICIES_URL}/{POLICY_ID}",  # Note: When we want to update a specific policy we use PUT and provide specific policy ID
        headers=headers,
        params=params,
        json=updated_policy_payload,
        verify=globals().get("VERIFY_SSL", True),
    )

    if response.status_code == 200:
        policy_response = response.json()
        print("Policy updated successfully")
        print(f"Policy ID: {policy_response['metadata']['id']}")
        print(f"Policy Name: {policy_response['entity']['name']}")
    else:
        print(f"Error updating policy: {response.status_code}")
        print(json.dumps(response.json(), indent=2))
        raise Exception(
            f"API call failed with status {response.status_code}: {response.text}"
        )

Policy updated successfully
Policy ID: 970c0d00-375a-45e4-b2e7-de3e82ef265b
Policy Name: Custom PII Detector


## Policy Enforcement with Built-in Detectors
This section demonstrates how to enforce policies using IBM's built-in detectors.
 We'll create two separate policies:


### HAP Detection with BLOCK Action
Create a policy that uses the HAP detector to block harmful content.

In [ ]:
# HAP Policy configuration
hap_policy_payload = {
    "name": "HAP Content Blocking Policy2",
    "description": "Policy to block hate, abuse, and profanity content",
    "block_message": "Your content contains inappropriate language and has been blocked. Please revise your message.",
    "mask_character": "*",
    "output": [],
    "input": [
        {
            "action": "block",
            "detector": "hap",
            "detector_properties": [{"name": "threshold", "value": "0.75"}],
        }
    ],
    "policy_status": "publish",
    "tags": ["hap", "content-safety", "blocking"],
}


params = {"inventory_id": INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=hap_policy_payload,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 201:
    hap_policy_response = response.json()
    HAP_POLICY_ID = hap_policy_response["metadata"]["id"]
    print("HAP Policy created successfully")
    print(f"Policy ID: {HAP_POLICY_ID}")
    print(f"Policy Name: {hap_policy_response['entity']['name']}")
else:
    print(f"Error creating HAP policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

HAP Policy created successfully
Policy ID: 3e9a2a58-ab74-4566-a24c-fde1eb2e4e0c
Policy Name: HAP Content Blocking Policy2


 ### Enforce HAP Policy - BLOCK Example

In [ ]:
if "HAP_POLICY_ID" in locals():

    hap_test_text = "I hate people from the moon"
    hap_enforce_payload = {
        "text": hap_test_text,
        "direction": "input",
        "detectors_properties": {"hap": {"threshold": "0.75"}},
    }

    print(f"Test Text: {hap_test_text}")
    params = {"inventory_id": INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{HAP_POLICY_ID}",
        headers=headers,
        params=params,
        json=hap_enforce_payload,
        verify=globals().get("VERIFY_SSL", True),
    )

    if response.status_code == 200:
        result = response.json()

        was_blocked = result["entity"]["text"] != hap_test_text
        print("Enforcement Result:")
        print(f"Status: {result['entity']['status']['overall']}")
        print(
            f"Detectors Run: {result['entity']['status']['summary']['total_detectors']}"
        )
        print(
            f"Detectors Succeeded: {result['entity']['status']['summary']['succeeded']}"
        )

        if was_blocked:
            print(f"Content was BLOCKED")
            print(f"Block Message: {result['entity']['text']}")
        else:
            print(f"Content was ALLOWED")
            print(f"Processed Text: {result['entity']['text']}")

Test Text: I hate people from the moon


### PII Detection with MASK Action
Create a policy that uses the PII detector to mask sensitive information.


In [ ]:
pii_policy_payload = {
    "name": "PII Data Masking Policy2",
    "description": "Policy to mask personally identifiable information",
    "block_message": "Content blocked due to policy violation.",
    "mask_character": "*",
    "output": [],
    "input": [
        {
            "action": "mask",
            "detector": "pii",
            "detector_properties": [{"name": "threshold", "value": "0.5"}],
        }
    ],
    "policy_status": "publish",
    "tags": ["pii", "data-privacy", "masking"],
}

params = {"inventory_id": INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=pii_policy_payload,
    verify=globals().get("VERIFY_SSL", True),
)

if response.status_code == 201:
    pii_policy_response = response.json()
    PII_POLICY_ID = pii_policy_response["metadata"]["id"]
    print("PII Policy created successfully")
    print(f"Policy ID: {PII_POLICY_ID}")
    print(f"Policy Name: {pii_policy_response['entity']['name']}")
else:
    print(f"Error creating PII policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))
    raise Exception(
        f"API call failed with status {response.status_code}: {response.text}"
    )

PII Policy created successfully
Policy ID: 0ddd7d96-9e2c-4cca-a09f-2378c5aa755b
Policy Name: PII Data Masking Policy2


### Enforce PII Policy - MASK Example


In [19]:
if "PII_POLICY_ID" in locals():

    pii_test_text = "My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street."
    pii_enforce_payload = {
        "text": pii_test_text,
        "direction": "output",
        "detectors_properties": {"pii": {"threshold": "0.5"}},
    }

    print(f"Test Text: {pii_test_text}")
    params = {"inventory_id": INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{PII_POLICY_ID}",
        headers=headers,
        params=params,
        json=pii_enforce_payload,
        verify=globals().get("VERIFY_SSL", True),
    )

    if response.status_code == 200:
        result = response.json()
        returned_text = result["entity"]["text"]

        was_masked = returned_text != pii_test_text

        print("Enforcement Result:")

        if was_masked:
            print(f"Overall Status: {result['entity']['status']['overall']}")
            print(
                f"Total Detectors: {result['entity']['status']['summary']['total_detectors']}"
            )
            print(f"Succeeded: {result['entity']['status']['summary']['succeeded']}")
            print(f"Failed: {result['entity']['status']['summary']['failed']}")
            print(f"PII was MASKED by the detector")
            print(f"Original Text: {pii_test_text}")
            print(f"Masked Text:   {returned_text}")
        else:
            print(f"No PII detected or masking applied")
            print(f"Returned Text: {returned_text}")
            
    else:
        print(f"Error enforcing policy: {response.status_code}")
        print(json.dumps(response.json(), indent=2))

Test Text: My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street.
Enforcement Result:
No PII detected or masking applied
Returned Text: My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street.
